# Part 3: Practical Data Preparation

**Objective:** Handle categorical features using One-Hot Encoding and address class imbalance using SMOTE.

## 1. Setup

Import necessary libraries.

In [51]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

## 2. Data Loading

Load the dataset.

In [52]:
def load_data(file_path):
    """
    Load the synthetic health data from a CSV file.
    
    Args:
        file_path: Path to the CSV file
        
    Returns:
        DataFrame containing the data
    """
    # Load the CSV file using pandas
    try:
        data = pd.read_csv(file_path, parse_dates=['timestamp'])
    except FileNotFoundError:
        print("File not exist!")
        data = None
    
    return data

## 3. Categorical Feature Encoding

Implement `encode_categorical_features` using `OneHotEncoder`.

In [53]:
def encode_categorical_features(df, column_to_encode='smoker_status', need_encoder=False):
    """
    Encode a categorical column using OneHotEncoder.
    
    Args:
        df: Input DataFrame
        column_to_encode: Name of the categorical column to encode
        
    Returns:
        DataFrame with the categorical column replaced by one-hot encoded columns
    """
    # 1. Drop the original column
    con_var = df.drop(columns=column_to_encode)

    # 2. Apply OneHotEncoder to the categorical column
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded = encoder.fit_transform(df[[column_to_encode]])  # Keep it 2D

    # 3. Get new column names
    new_column_names = encoder.get_feature_names_out([column_to_encode])

    # 4. Create a DataFrame with encoded features
    encoded_df = pd.DataFrame(encoded, columns=new_column_names, index=df.index)

    # 5. Concatenate and return
    if need_encoder:
        return pd.concat([con_var, encoded_df], axis=1), encoder
    else:
        return pd.concat([con_var, encoded_df], axis=1)

## 4. Data Preparation

Implement `prepare_data_part3` to handle the train/test split correctly.

In [54]:
def prepare_data_part3(df, test_size=0.2, random_state=42):
    """
    Prepare data with categorical encoding.

    Args:
        df: Input DataFrame
        test_size: Proportion of data for testing
        random_state: Random seed for reproducibility

    Returns:
        X_train, X_test, y_train, y_test
    """
    # 1. Encode categorical features
    df_encoded, encoder = encode_categorical_features(df, need_encoder=True)

    # 2. Define features and target
    feature_columns = df_encoded.columns.drop(['patient_id', 'timestamp', 'disease_outcome'])
    X = df_encoded[feature_columns]
    y = df_encoded['disease_outcome']

    # 3. Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 4. Handle missing values
    imputer = SimpleImputer(strategy='mean')
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)

    # 5. Restore DataFrames with proper column names
    X_train = pd.DataFrame(X_train_imputed, columns=feature_columns, index=X_train.index)
    X_test = pd.DataFrame(X_test_imputed, columns=feature_columns, index=X_test.index)

    return X_train, X_test, y_train, y_test, encoder

## 5. Handling Imbalanced Data

Implement `apply_smote` to oversample the minority class.

In [55]:
def apply_smote(X_train, y_train, random_state=42):
    """
    Apply SMOTE to oversample the minority class.
    
    Args:
        X_train: Training features
        y_train: Training target
        random_state: Random seed for reproducibility
        
    Returns:
        Resampled X_train and y_train with balanced classes
    """
    # YOUR CODE HERE
    # Apply SMOTE to balance the classes
    smote = SMOTE(random_state=random_state)
    X_train_new, y_train_new = smote.fit_resample(X_train, y_train)
    
    # Placeholder return - replace with your implementation
    return X_train_new, y_train_new

## 6. Model Training and Evaluation

Train a model on the SMOTE-resampled data and evaluate it.

In [56]:
def train_logistic_regression(X_train, y_train):
    """
    Train a logistic regression model.
    
    Args:
        X_train: Training features
        y_train: Training target
        
    Returns:
        Trained logistic regression model
    """
    # YOUR CODE HERE
    # Initialize and train a LogisticRegression model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    return model  # Replace with actual implementation

def calculate_evaluation_metrics(model, X_test, y_test):
    """
    Calculate classification evaluation metrics.
    
    Args:
        model: Trained model
        X_test: Test features
        y_test: Test target
        
    Returns:
        Dictionary containing accuracy, precision, recall, f1, auc, and confusion_matrix
    """
    # YOUR CODE HERE
    # 1. Generate predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    # 2. Calculate metrics: accuracy, precision, recall, f1, auc
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    # 3. Create confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    # 4. Return metrics in a dictionary
    
    # Placeholder return - replace with your implementation
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc,
        'confusion_matrix': cm
    }

## 7. Save Results

Save the evaluation metrics to a text file.

In [57]:
# YOUR CODE HERE
# 1. Create 'results' directory if it doesn't exist
# 2. Format metrics as strings
# 3. Write metrics to 'results/results_part3.txt'

## 8. Main Execution

Run the complete workflow.

In [ ]:
# Main execution
if __name__ == "__main__":
    # 1. Load data
    data_file = 'data/synthetic_health_data.csv'
    df = load_data(data_file)
    
    # 2. Prepare data with categorical encoding
    X_train, X_test, y_train, y_test, _ = prepare_data_part3(df)
    
    # 3. Apply SMOTE to balance the training data
    X_train_resampled, y_train_resampled = apply_smote(X_train, y_train)
    
    # 4. Train model on resampled data
    model = train_logistic_regression(X_train_resampled, y_train_resampled)
    
    # 5. Evaluate on original test set
    metrics = calculate_evaluation_metrics(model, X_test, y_test)
    
    # 6. Print metrics
    for metric, value in metrics.items():
        if metric != 'confusion_matrix':
            print(f"{metric}: {value:.4f}")
    
    # 7. Save results
    # (Your code for saving results)
    os.makedirs('results', exist_ok=True)
    results_file = 'results/results_part3.txt'
    with open(results_file, 'w') as f:
        for metric, value in metrics.items():
            if metric != 'confusion_matrix':
                f.write(f"{metric}: {value:.4f}\n")
    
    # 8. Load Part 1 results for comparison
    import json
    try:
        part1_metrics = {}
        with open('results/results_part1.txt', 'r') as f:
            for line in f:
                if ':' in line:
                    key, value = line.strip().split(':', 1)
                    part1_metrics[key.strip()] = float(value.strip())

        # 9. Compare models
        comparison = compare_models(part1_metrics, metrics)
        print("\nModel Comparison (improvement percentages):")
        for metric, improvement in comparison.items():
            print(f"{metric}: {improvement:.2f}%")
    except FileNotFoundError:
        print("Part 1 results not found. Run part1_introduction.ipynb first.")

accuracy: 0.8568
precision: 0.3930
recall: 0.8601
f1: 0.5395
auc: 0.9263
{'accuracy': 0.9168, 'precision': 0.6615, 'recall': 0.3007, 'f1': 0.4135, 'auc': 0.9084}

Model Comparison (improvement percentages):
accuracy: -6.55%
precision: -40.59%
recall: 186.05%
f1: 30.47%
auc: 1.97%


## 9. Compare Results

Implement a function to compare model performance between balanced and imbalanced data.

In [42]:
def compare_models(part1_metrics, part3_metrics):
    """
    Calculate percentage improvement between models trained on imbalanced vs. balanced data.
    
    Args:
        part1_metrics: Dictionary of metrics from Part 1
        part3_metrics: Dictionary of metrics from Part 3
        
    Returns:
        Dictionary with metric names as keys and improvement percentages as values
    """
    improvement = {}
    for key in part1_metrics:
        # Skip metrics not in both or with zero baseline
        if key in part3_metrics and part1_metrics[key] != 0:
            part1_value = part1_metrics[key]
            part3_value = part3_metrics[key]
            percent_change = ((part3_value - part1_value) / part1_value) * 100
            improvement[key] = percent_change
        else:
            improvement[key] = 0.0  # Or handle this differently if needed
    return improvement
